# Fast Prompting en Acción: Prevención de quiebres de stock en el área analítica del laboratorio

**Autor:** Matías Drago
**Curso:** IA - Generación de Prompts (Coderhouse) — Comisión 96165
**Entrega #2:** Fast Prompting en Acción — Desentrañando la Magia

Este notebook documenta la aplicación práctica de técnicas de Fast Prompting (Zero-shot, One-shot y Few-shot) sobre el problema seleccionado en la Preentrega 1: la prevención de quiebres de stock de reactivos e insumos en el área analítica del laboratorio.

Los prompts fueron ejecutados de forma manual en **ChatGPT**, utilizando datos reales de consumo extraídos de los reportes "Contador de prueba detallado" del equipo Cobas Pure (mayo, junio y julio de 2026).

## 1. Datos de partida

A partir de tres reportes del analizador (12/05/2026, 02/06/2026 y 01/07/2026) se calculó el **ritmo de consumo diario** de cada reactivo, comparando el total acumulado de determinaciones entre cortes. Sobre esa base se definió un **stock hipotético** para simular el escenario de alerta temprana, dado que no se contó con el dato de stock físico real al momento de la entrega.

In [ ]:
import pandas as pd

data = {
    "Reactivo":      ["ISE NA", "CREJ2", "GLUC3", "CHOL2-A", "TRIGL", "TSH"],
    "Stock_actual":  [850, 3200, 1400, 6000, 500, 200],
    "Ritmo_consumo_dia": [108.8, 101.1, 93.7, 63.8, 9.6, 0.0],
}

df = pd.DataFrame(data)
df["Dias_restantes"] = df.apply(
    lambda row: round(row["Stock_actual"] / row["Ritmo_consumo_dia"], 1) if row["Ritmo_consumo_dia"] > 0 else None,
    axis=1
)
df

## 2. Criterio de clasificación por zona de riesgo

| Zona | Días de cobertura |
|---|---|
| 🔴 Rojo | menos de 10 días |
| 🟡 Amarillo | entre 10 y 30 días |
| 🟢 Verde | más de 30 días |
| Sin riesgo | ritmo de consumo = 0 (stock estático, no se agota por consumo) |

Este criterio se le explicitó al modelo en el **one-shot** y el **few-shot**, pero **no** en el **zero-shot** — lo cual, como se verá, generó una diferencia de resultado.

## 3. Prompt — Técnica Zero-shot

Sin ejemplos previos, se le pide directamente al modelo que calcule los días restantes y clasifique el riesgo, dejando que él mismo defina (o infiera) el criterio de corte.

**Prompt utilizado:**

```
Tenés la siguiente tabla de stock de reactivos de laboratorio, con el stock actual
en determinaciones disponibles y el ritmo de consumo diario:

Reactivo | Stock actual | Ritmo consumo/día
ISE NA | 850 | 108.8
CREJ2 | 3200 | 101.1
GLUC3 | 1400 | 93.7
CHOL2-A | 6000 | 63.8
TRIGL | 500 | 9.6
TSH | 200 | 0.0

Para cada reactivo, calculá en cuántos días se agotará el stock y clasificalo en
una zona de riesgo (verde, amarillo o rojo).
```

**Resultado obtenido (ChatGPT):**

La fórmula aplicada fue Días de cobertura = Stock actual ÷ Ritmo de consumo diario, con criterio de riesgo autodefinido por el modelo (🟢 Verde >30 días, 🟡 Amarillo 15-30 días, 🔴 Rojo <15 días; consumo 0 → sin riesgo por consumo).

| Reactivo | Stock actual | Consumo/día | Días hasta agotarse | Zona |
|---|---|---|---|---|
| ISE NA | 850 | 108.8 | 7.8 días | 🔴 Rojo |
| CREJ2 | 3200 | 101.1 | 31.7 días | 🟢 Verde |
| GLUC3 | 1400 | 93.7 | 14.9 días | 🔴 Rojo |
| CHOL2-A | 6000 | 63.8 | 94.0 días | 🟢 Verde |
| TRIGL | 500 | 9.6 | 52.1 días | 🟢 Verde |
| TSH | 200 | 0.0 | No se agota | 🟢 Verde |

El modelo agregó además una sección de **"Prioridad de acción"** (reposición urgente/prioritaria/monitorear/normal) y una observación propia sobre el caso límite de GLUC3, sugiriendo una categoría adicional de "alerta preventiva" para valores cercanos al umbral.

## 4. Prompt — Técnica One-shot

Se fija explícitamente el criterio de clasificación mostrando un ejemplo ya resuelto, para que el modelo replique el mismo umbral en el resto de los casos.

**Prompt utilizado:**

```
Te doy un ejemplo de cómo clasificar reactivos según su stock y ritmo de consumo:

Ejemplo: Reactivo "UREAL", stock 2000, ritmo 99.8/día → Días restantes: 20.0 →
Zona: AMARILLA (criterio: rojo <10 días, amarillo 10-30 días, verde >30 días)

Ahora aplicá el mismo criterio a esta tabla:
Reactivo | Stock actual | Ritmo consumo/día
ISE NA | 850 | 108.8
CREJ2 | 3200 | 101.1
GLUC3 | 1400 | 93.7
CHOL2-A | 6000 | 63.8
TRIGL | 500 | 9.6
TSH | 200 | 0.0
```

**Resultado obtenido (ChatGPT):**

| Reactivo | Stock actual | Ritmo consumo/día | Días restantes | Zona |
|---|---|---|---|---|
| ISE NA | 850 | 108.8 | 7.8 | ROJA |
| CREJ2 | 3200 | 101.1 | 31.7 | VERDE |
| GLUC3 | 1400 | 93.7 | 14.9 | AMARILLA |
| CHOL2-A | 6000 | 63.8 | 94.0 | VERDE |
| TRIGL | 500 | 9.6 | 52.1 | VERDE |
| TSH | 200 | 0.0 | Indefinido (sin consumo) | VERDE (stock estático) |

A diferencia del zero-shot, acá el modelo **replicó exactamente** el criterio del ejemplo (rojo <10, amarillo 10-30, verde >30), clasificando a GLUC3 (14.9 días) como AMARILLA en lugar de ROJA.

## 5. Prompt — Técnica Few-shot

Se refuerza el criterio con tres ejemplos, uno por cada zona de riesgo, para maximizar la consistencia del modelo al aplicar el umbral.

**Prompt utilizado:**

```
Estos son ejemplos de clasificación de reactivos por zona de riesgo
(rojo <10 días, amarillo 10-30 días, verde >30 días):

Ejemplo 1: "BILT3", stock 400, ritmo 62.2/día → 6.4 días → ZONA ROJA
Ejemplo 2: "MG2", stock 900, ritmo 21.2/día → 42.5 días → ZONA VERDE
Ejemplo 3: "UA2", stock 250, ritmo 11.9/día → 21.0 días → ZONA AMARILLA

Aplicá el mismo criterio de clasificación a esta tabla, mostrando el cálculo de
días restantes para cada uno:
Reactivo | Stock actual | Ritmo consumo/día
ISE NA | 850 | 108.8
CREJ2 | 3200 | 101.1
GLUC3 | 1400 | 93.7
CHOL2-A | 6000 | 63.8
TRIGL | 500 | 9.6
TSH | 200 | 0.0
```

**Resultado obtenido (ChatGPT):**

| Reactivo | Stock actual | Ritmo consumo/día | Cálculo (Stock / Ritmo) | Días restantes | Zona |
|---|---|---|---|---|---|
| ISE NA | 850 | 108.8 | 850 / 108.8 | 7.8 días | ZONA ROJA |
| CREJ2 | 3200 | 101.1 | 3200 / 101.1 | 31.7 días | ZONA VERDE |
| GLUC3 | 1400 | 93.7 | 1400 / 93.7 | 14.9 días | ZONA AMARILLA |
| CHOL2-A | 6000 | 63.8 | 6000 / 63.8 | 94.0 días | ZONA VERDE |
| TRIGL | 500 | 9.6 | 500 / 9.6 | 52.1 días | ZONA VERDE |
| TSH | 200 | 0.0 | Sin consumo | Indefinido | ZONA VERDE |

El few-shot replicó correctamente el criterio (igual que el one-shot) y además mostró explícitamente el cálculo paso a paso para cada fila, aportando mayor trazabilidad al resultado.

In [ ]:
# Comparación de resultados entre las 3 técnicas para el caso límite (GLUC3, 14.9 días)

comparacion = pd.DataFrame({
    "Técnica": ["Zero-shot", "One-shot", "Few-shot"],
    "Criterio de umbral aplicado": ["Autodefinido por el modelo (rojo<15 / amarillo 15-30)",
                                     "Dado por el ejemplo (rojo<10 / amarillo 10-30)",
                                     "Dado por los 3 ejemplos (rojo<10 / amarillo 10-30)"],
    "Clasificación de GLUC3 (14.9 días)": ["🔴 Rojo", "🟡 Amarillo", "🟡 Amarillo"],
    "Consistencia con el criterio pedido": ["Baja — el modelo infirió su propio umbral",
                                             "Alta — replicó el umbral del ejemplo",
                                             "Alta — replicó el umbral y mostró el cálculo paso a paso"],
})
comparacion

## 6. Análisis comparativo

El experimento evidencia un punto central para la viabilidad del proyecto: **cuando el criterio de riesgo no se fija explícitamente en el prompt (zero-shot), el modelo puede inferir su propio umbral**, lo que introduce inconsistencia en la clasificación. Esto se vio claramente en el reactivo GLUC3 (14.9 días de cobertura), clasificado como 🔴 Rojo en el zero-shot pero como 🟡 Amarillo en el one-shot y el few-shot.

El **one-shot** y el **few-shot**, al fijar el criterio mediante ejemplos, lograron resultados consistentes entre sí. El few-shot, además, aportó un plus de trazabilidad al mostrar el cálculo explícito por fila, lo cual sería valioso en un sistema real donde el equipo del laboratorio necesita poder verificar cómo se llegó a cada alerta.

**Conclusión:** para este caso de uso, se recomienda avanzar con la técnica **few-shot**, ya que combina consistencia del criterio con transparencia en el cálculo — dos atributos clave para que el sistema de alertas sea confiable y auditable en un entorno de laboratorio clínico.

## 7. ¿Estas técnicas mejoran la propuesta de la Preentrega 1?

Sí. La Preentrega 1 planteaba el prompt de la Etapa 1 de forma genérica ("prompt que calcule el ritmo de consumo y estime en cuántos días se agota cada insumo"), sin especificar cómo evitar la ambigüedad del criterio de riesgo. Este ejercicio con Fast Prompting demuestra en la práctica que **definir el criterio mediante ejemplos (few-shot) es necesario**, no opcional, para que el resultado sea confiable y reproducible — un hallazgo que fortalece la justificación de viabilidad técnica del proyecto.